# 03. Vectors and 2D maps

**Atlas source reading copy · reference-only**

[Course guide](index.ipynb) · [Pinned original](https://github.com/jltobias/JupyterLite-terrio/blob/18574d4e7a8dabb99d3db65b20c6180185578482/notebooks/03-vectors-2d-maps.ipynb)



Reading/editing copy only in the shared Atlas kernel. TerrIO 0.9.0 uses a pyemscripten_2025_0 wasm wheel for Pyodide 0.29; the source pins jupyterlite-pyodide-kernel 0.8.0b0 (default Pyodide 0.29.4). This wheel is incompatible with Atlas Pyodide 314; do not install it here or downgrade the shared runtime. [Run this lesson in its matched source Lab](https://jltobias.github.io/JupyterLite-terrio/lite/lab/index.html?path=03-vectors-2d-maps.ipynb). The source uses Plotly 6.3.1, xarray 2025.10.1 and optional CDN/WebGL graphics. Natural Earth countries are real generalized geography; all other numerical fixtures are explicitly synthetic. Native HTTP range I/O, specialized drivers and real service/model calls remain optional external continuations.

**Provenance:** `notebooks/03-vectors-2d-maps.ipynb` at `18574d4e7a8dabb99d3db65b20c6180185578482`; original SHA-256 `eefde9c5831154c74c8d39fafd3df9f9a16c0a3514ca852f6f9a305057fb672c`. Original code MIT; original teaching prose, synthetic data and chapter SVG diagrams CC0. Natural Earth is public domain; TerrIO/dependency/EPSG terms remain separately scoped in retained notices. No TerrIO wheel redistributed.

**Verification:** imported by static inspection; saved upstream outputs cleared. Runtime compatibility is separate from executed verification. See the course guide for dependencies, excluded assets, aliases and original credits.


![Chapter 3: Vectors and 2D maps; abstract contour illustration](assets/03-vectors-2d-maps.svg)

# 03. Vectors and 2D maps

Explore a public-domain world

**Learning goals:** Read Natural Earth features; filter attributes with TerrIO; make an interactive outline map.

**Time:** 30–45 minutes. **Prerequisites:** basic Python arrays; earlier chapters are helpful but every notebook runs independently.

**Run:** choose the Python (Pyodide) kernel in JupyterLite, then **Run → Run All Cells**. First use downloads the runtime and packages. Native Python users install the repository requirements first. Plotly views need network access to its versioned JavaScript CDN and 3D views need WebGL; static figures and numerical summaries remain available.

[Open this chapter in JupyterLite](https://jltobias.github.io/JupyterLite-terrio/lite/lab/index.html?path=03-vectors-2d-maps.ipynb) · [Learning resources](https://jltobias.github.io/JupyterLite-terrio/book/resources.html)

**Data:** all generated measurements are synthetic. Natural Earth outlines, where used, are public domain. Generated files go to `outputs/`; download them to retain them outside this browser.


In [ ]:
import sys
if sys.platform == "emscripten":
    import piplite
    await piplite.install(["terrio==0.9.0", "matplotlib", "plotly==6.3.1", "xarray==2025.10.1"])
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import terrio
from terrio_lab import (DATA, OUTPUT, CRS, TRANSFORM, CREDIT, terrain, raster,
                        countries, rings, map_outlines, image_grid, interactive, surface, globe)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print("Ready: TerrIO", terrio.__version__, "|", "browser" if sys.platform == "emscripten" else "native Python")

## Start with provenance

Natural Earth 1:110m Admin 0 Countries v5.1.2 is a public-domain cartographic dataset by Tom Patterson, Nathaniel Vaughn Kelso, and contributors. Its generalized boundaries are for small-scale mapping, not legal boundary adjudication. The bundled original file and SHA-256 are recorded in data/sources.json.

In [ ]:
layer = terrio.open_vector(str(DATA / "countries.geojson")).layers[0]
world = layer.to_geojson()
print("Features:", len(layer), "geometry:", layer.geometry_type)
print("Example name:", world["features"][0]["properties"]["ADMIN"])
assert len(world["features"]) == 177
map_outlines(world)
plt.show()

## Filter, then inspect the result

SQL-style attribute filters select records; they do not clip geometry at a region boundary. The CONTINENT field follows this dataset's classification. Do not substitute that classification for every cultural or political definition of a continent.

In [ ]:
dest = OUTPUT / "africa.geojson"
terrio.convert(str(DATA / "countries.geojson"), str(dest), where="CONTINENT = 'Africa'")
africa = terrio.open_vector(str(dest)).layers[0].to_geojson()
assert all(f["properties"]["CONTINENT"] == "Africa" for f in africa["features"])
fig, ax = map_outlines(africa, "Natural Earth features classified as Africa")
ax.set(xlim=(-25, 60), ylim=(-40, 40))
plt.show()

## Compare feature counts

Counts describe the records in a generalized map. They are not estimates of land area, population, or national status. In particular, a multipart geometry can hold several islands within one record.

In [ ]:
from collections import Counter
counts = Counter(f["properties"]["CONTINENT"] for f in world["features"])
fig, ax = plt.subplots(figsize=(8, 4))
labels, values = zip(*sorted(counts.items(), key=lambda item: item[1]))
ax.barh(labels, values, color="#008b8b")
ax.set(xlabel="Feature count", title="Records by Natural Earth CONTINENT field")
fig.tight_layout(); plt.show()

## Pan, zoom, and inspect

This interactive 2D view draws the bundled outlines without fetching basemap tiles. Hover over an outline for its country name, drag to zoom, and double-click to reset. Longitude/latitude axes make its simple equirectangular treatment visible.

In [ ]:
import plotly.graph_objects as go
fig = go.Figure()
for feature in world["features"]:
    for ring in rings({"features": [feature]}):
        fig.add_trace(go.Scatter(x=ring[:, 0], y=ring[:, 1], mode="lines", name=feature["properties"]["ADMIN"],
                                line=dict(width=1, color="#008b8b"), showlegend=False,
                                hovertemplate=feature["properties"]["ADMIN"]+"<extra></extra>"))
fig.update_layout(title="2D map · Made with Natural Earth", xaxis_title="Longitude (°)", yaxis_title="Latitude (°)")
interactive(fig)

## Your investigation

Filter a second continent, report its feature count, and explain why feature count is not a measure of geographic size.

### Discussion guide

Use the same CONTINENT filter with a value present in counts. Countries and multipart island groups each occupy records with very different areas.

## Sources and credit

- [Source 1](https://www.naturalearthdata.com/about/terms-of-use/)
- [Source 2](https://github.com/nvkelso/natural-earth-vector/tree/v5.1.2)
- [Source 3](https://terrio.dev/sql/)

TerrIO: Qiusheng Wu, Mavrick Fitzgerald, Guillermo Morales Sanchez, and contributors (2026). *terrio: Pure-Rust geospatial raster, vector and point cloud data I/O*, v0.9.0. [Zenodo concept DOI](https://doi.org/10.5281/zenodo.23642204). The DOI identifies the project; the execution version is pinned separately.

Natural Earth: Tom Patterson, Nathaniel Vaughn Kelso, and contributors, v5.1.2, [public-domain terms](https://www.naturalearthdata.com/about/terms-of-use/). Original explanations, synthetic data, code, and contour title card: JupyterLite-terrio contributors. Software and underlying third-party notices are documented in the repository's attribution page. Consult that page before redistributing dependency bundles.